### 微调

#### 指令数据集准备

In [ ]:
import json
from pathlib import Path


# =========================
# 配置区
# =========================
def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "wsd_train.json").exists():
            return candidate
    raise FileNotFoundError("未找到 data/wsd_train.json")


REPO_ROOT = find_repo_root()
INPUT_PATH = REPO_ROOT / "data" / "wsd_train.json"
OUTPUT_PATH = REPO_ROOT / "data" / "wsd_lora_train.json"


# =========================
# system prompt
# =========================
SYSTEM_PROMPT = """你是一位古汉语词义消歧专家。请根据古文上下文，从给定候选义项中选择目标字的正确义项。只输出义项 ID（如 s1），不要输出解释。"""


# =========================
# 构造 input
# =========================
def build_input(sample):
    text = sample["text"]
    word = sample["word"]
    options = sample["options"]
    option_lines = [f"{option_id}: {meaning}" for option in options for option_id, meaning in option.items()]

    return f'古文：{text}\n目标字：{word}\n候选义项：\n' + "\n".join(option_lines)


# =========================
# 主流程
# =========================
def main():
    # 1. 读取数据
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"加载数据量: {len(data)}")

    # 2. 构建 LoRA 数据
    lora_dataset = []

    for item in data:
        output = item.get("label")

        # 过滤无效数据
        if not output:
            continue

        example = {
            "instruction": SYSTEM_PROMPT,
            "input": build_input(item),
            "output": output
        }

        lora_dataset.append(example)

    print(f"最终可用数据量: {len(lora_dataset)}")

    # 3. 保存
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(lora_dataset, f, ensure_ascii=False, indent=2)

    print(f"[OK] 已保存到: {OUTPUT_PATH}")


# =========================
# 入口
# =========================
if __name__ == "__main__":
    main()

修改 `/root/LLaMA-Factory/data/dataset_info.json` 文件: 添加一个指令数据集的描述，如下
```json
  "wsd_train": {
    "file_name": "/path/to/DynaSense/data/wsd_lora_train.json"
  },
```
键将作为后续配置文件的数据集名称使用

#### 微调参数文件准备

当前保存在 `/root/LLaMA-Factory/examples/train_lora/Qwen3-1.5B-Instruct.yaml`

```yaml
### model
# 手动下载的需要放绝对路径，需要自行修改
# 修改下面这一行/root/autodl-tmp/models/Qwen3-1.5B-Instruct的路径就行
model_name_or_path: /root/autodl-tmp/models/Qwen3-1.5B-Instruct
trust_remote_code: true


### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.1
lora_target: all
deepspeed: /root/LLaMA-Factory/examples/deepspeed/ds_z0_config.json  # choices: [ds_z0_config.json, ds_z2_config.json, ds_z3_config.json]

### dataset
# dataset: identity,alpaca_en_demo
dataset: wsd_train

# template 不成功的清按详情参照https://github.com/hiyouga/LlamaFactory/blob/main/README_zh.md#%E8%AE%AD%E7%BB%83%E6%96%B9%E6%B3%95abs
# 下一行为当前需要使用的模板，{模型类别}:{template}
# qwen3:qwen3; qwen2.5:qwen; gemma:gemma; llama2:llama2; llama3:llama3; gpt2:gpt_oss; deepseek-distill: deepseekr1
template: qwen
cutoff_len: 2048
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16
dataloader_num_workers: 4


### output
# 这里是训练完成后的权重和部分中间过程的内容，需要自行修改
output_dir: /root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none  # choices: [none, wandb, tensorboard, swanlab, mlflow]


### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 2
learning_rate: 1.0e-4
num_train_epochs: 30.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000
resume_from_checkpoint: null
```

#### 训练命令

`/root/LLaMA-Factory/examples/train_lora/1-Qwen3-1.5B-Instruct.yaml`就修改成训练的配置文件路径即可

FORCE_TORCHRUN=1 llamafactory-cli train /root/LLaMA-Factory/examples/train_lora/1-Qwen3-1.5B-Instruct.yaml

训练完成的权重保存在 `/root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora`

#### 合并模型和适配器

配置文件: `LLaMA-Factory/examples/merge_lora/Qwen3-1.5B-Instruct.yaml`

```yaml
### Note: DO NOT use quantized model or quantization_bit when merging lora adapters

### model
# 原始模型的导出路径，需要自行修改
model_name_or_path: /root/autodl-tmp/models/Qwen3-1.5B-Instruct
# lora权重的导出路径，需要自行修改
adapter_name_or_path: /root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora
# 不同模型的模板，需要自行修改
template: qwen
trust_remote_code: true

### export
# 合并模型的导出路径，需要自行修改
export_dir: /root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora_models
export_size: 2
export_device: auto  # choices: [cpu, auto]
export_legacy_format: false
```

运行命令

`/root/LLaMA-Factory/examples/merge_lora/Qwen3-1.5B-Instruct.yaml`前面**合并模型和适配器**的配置文件，需要自行修改

llamafactory-cli export /root/LLaMA-Factory/examples/merge_lora/Qwen3-1.5B-Instruct.yaml

#### 启动模型

##### xinference (本次使用)

如果需要单独的模型，则需要注册-配置-启动一条龙